In [1]:
import pandas as pd

from db import fetch_rows, fetch_df, fetch_table_rows, fetch_table_df, execute_write

```python
# Reference Functions
fetch_rows(query, params=None)  # Execute any query and return rows as a list of dicts
fetch_table_rows(table_name, limit=None)  # Fetch rows from a table with an optional limit
fetch_table_df(table_name, limit=None)  # Fetch df of any table with an optional limit
execute_write(query, params=None)  # Execute any write query (INSERT, UPDATE, DELETE) with optional parameters. Returns None.
```

In [2]:
# # delete all data from all tables and reset auto-incrementing ids
# # Dangerous reset helper example.
# execute_write("""
# TRUNCATE TABLE 
# questions,
# users,
# subjects,
# topics,
# reviews,
# badges,
# stats,
# settings,
# user_subjects,
# user_badges,
# sync_meta
# RESTART IDENTITY CASCADE;
# """)


In [16]:
# schema info
schema_df = fetch_df("""
SELECT
    conrelid::regclass AS table_name,
    conname AS constraint_name,
    pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE contype IN ('p','u')
ORDER BY table_name;
""")
schema_df.head(2)

,table_name,constraint_name,definition
0,pg_default_acl,pg_default_acl_role_nsp_obj_index,"UNIQUE (defaclrole, defaclnamespace, defaclobj..."
1,pg_default_acl,pg_default_acl_oid_index,PRIMARY KEY (oid)


In [30]:
# pg_stat_user_tables content and table name, row counts
pg_stat_user_tables_df = fetch_table_df("pg_stat_user_tables", limit=None)
pg_stat_user_tables_df.rename(columns={"relname": "table_name", "n_live_tup": "row_count"}, inplace=True)
pg_stat_user_tables_df[["table_name", "row_count"]].sort_values("table_name").reset_index(drop=True)

,table_name,row_count
0,badges,0
1,questions,3738
2,reviews,985
3,settings,105
4,stats,1459
5,subjects,3
6,sync_meta,0
7,topics,23
8,user_badges,14
9,user_subjects,3


In [31]:
# badges table content
badges_df = fetch_table_df("badges", limit=None)
badges_df.head(2)

""


In [20]:
# Questions table content
Questions_table = fetch_df("""
SELECT *
FROM questions
""")
Questions_table.head(2)

,id,topic_id,type,question,answer
0,1,6,math-addition,4 + 3 = ?,7
1,2,6,math-addition,9 + 6 = ?,15


In [32]:
# reviews table content
reviews_df = fetch_table_df("reviews", limit=None)
reviews_df.head(2)

,user_id,question_id,repetition,interval,ease_factor,next_review,last_result,rev_id,last_modified_rev,sync_version,updated_at
0,1,100310,1,1,2.5,1775548168278,good,1775833626151,1775833626151,2,2026-04-10 20:37:05.992
1,1,100321,1,1,2.5,1775548472542,good,1775833626162,1775833626162,2,2026-04-10 20:37:05.992


In [19]:
# Settings table content
Settings_table = fetch_table_df("settings", 150).sort_values(["user_id", "key"]).reset_index(drop=True)
Settings_table.head(2)

,user_id,key,value,updated_at,sync_version
0,0,admin_selected_topic_path_1:2,11,2026-04-10 20:37:53.699,2
1,0,admin_selected_topic_path_1:3,21,2026-04-09 02:25:01.760,1


In [35]:
# stats table content
stats_df = fetch_table_df("stats", 150).sort_values(["user_id", "topic_id", "question_id", "wrong"]).reset_index(drop=True)
stats_df.head(2)

,id,user_id,question_id,topic_id,correct,wrong,practiced_at,updated_at
0,17015,1,100351,2,1,0,2026-04-06 12:24:27.824,2026-04-06 12:24:27.824
1,17016,1,100352,2,1,0,2026-04-06 12:24:59.064,2026-04-06 12:24:59.064


In [36]:
# subjects table content
subjects_df = fetch_table_df("subjects", limit=None)
subjects_df.head(2)

,id,name
0,1,Mathematics
1,2,English


In [37]:
# sync_meta table content
sync_meta_df = fetch_table_df("sync_meta", limit=None)
sync_meta_df.head(2)

""


In [38]:
# topics table content
topics_df = fetch_table_df("topics", limit=None)
topics_df.head(2)

,id,subject_id,parent_topic_id,key,name
0,1,1,NaN,multiplication_tables,Multiplication Tables
1,2,1,1.0,tables_1_5,Tables 1-5


In [40]:
# user_badges table content
user_badges_df = fetch_table_df("user_badges", limit=None)
user_badges_df.head(2)

,user_id,badge_id,unlocked_at,updated_at,sync_version
0,3,topic_master,2026-04-06 04:46:14.036,2026-04-08 11:56:03.808,43
1,3,100_questions,2026-04-09 11:36:59.603,2026-04-09 11:37:22.461,3


In [41]:
# user_subjects table content
user_subjects_df = fetch_table_df("user_subjects", limit=None)
user_subjects_df.head(2)

,user_id,subject_id
0,1,1
1,2,1


In [45]:
# users table content
users_df = fetch_table_df("users", limit=None)
users_df

,id,name,created_at
0,1,Bhavi,2026-04-09 10:52:32.799927
1,2,Madhu,2026-04-09 10:52:32.799927
2,3,Quiz Kid,2026-04-09 10:52:32.799927


#### Wrong answered questions (DND)

In [15]:
wrong_ans_ques_df = fetch_df("""
WITH latest_stats AS (
  SELECT DISTINCT ON (s.user_id, s.question_id)
    s.user_id,
    s.question_id,
    s.topic_id,
    s.practiced_at
  FROM stats s
  ORDER BY s.user_id, s.question_id, s.practiced_at DESC
),
wrong_reviews AS (
  SELECT
    r.user_id,
    r.question_id,
    r.last_result,
    r.updated_at
  FROM reviews r
  WHERE r.last_result = 'again'
),
enriched AS (
  SELECT
    wr.user_id,
    u.name AS user_name,
    wr.question_id,
    wr.last_result,
    wr.updated_at,
    COALESCE(ls.topic_id, q.topic_id) AS effective_topic_id,
    q.question AS q_question,
    q.answer   AS q_answer
  FROM wrong_reviews wr
  LEFT JOIN users u
    ON u.id = wr.user_id
  LEFT JOIN latest_stats ls
    ON ls.user_id = wr.user_id
   AND ls.question_id = wr.question_id
  LEFT JOIN questions q
    ON q.id::text = wr.question_id
)
SELECT
  e.user_name,
  s.name AS subject,
  COALESCE(tp.name, t.name) AS topic_level1,
  CASE WHEN tp.id IS NULL THEN NULL ELSE t.name END AS topic_level2,
  e.question_id,
  COALESCE(
    e.q_question,
    CASE
      WHEN e.question_id ~ '^[0-9]+$'
       AND right(e.question_id, 2) = '10'
       AND length(e.question_id) > 2
      THEN
        ((substring(e.question_id from 1 for length(e.question_id)-2))::int)::text
        || ' x 10'
      WHEN e.question_id ~ '^[0-9]+$'
       AND length(e.question_id) > 1
      THEN
        ((substring(e.question_id from 1 for length(e.question_id)-1))::int)::text
        || ' x ' ||
        (right(e.question_id, 1)::int)::text
      ELSE NULL
    END
  ) AS question,
  COALESCE(
    e.q_answer,
    CASE
      WHEN e.question_id ~ '^[0-9]+$'
       AND right(e.question_id, 2) = '10'
       AND length(e.question_id) > 2
      THEN
        (
          (substring(e.question_id from 1 for length(e.question_id)-2))::int
          * 10
        )::text
      WHEN e.question_id ~ '^[0-9]+$'
       AND length(e.question_id) > 1
      THEN
        (
          (substring(e.question_id from 1 for length(e.question_id)-1))::int
          * (right(e.question_id, 1)::int)
        )::text
      ELSE NULL
    END
  ) AS answer,
  e.last_result,
  e.updated_at
FROM enriched e
LEFT JOIN topics t
  ON t.id = e.effective_topic_id
LEFT JOIN topics tp
  ON tp.id = t.parent_topic_id
LEFT JOIN subjects s
  ON s.id = t.subject_id
ORDER BY e.user_name, subject, topic_level1, topic_level2, e.updated_at DESC;
""")
wrong_ans_ques_df.head(2)

,user_name,subject,topic_level1,topic_level2,question_id,question,answer,last_result,updated_at
0,Bhavi,English,Word Spellings,3 Letter Words,104005,I got a good grade on the test.,grade,again,2026-04-10 20:59:03.873
1,Bhavi,Mathematics,Multiplication Tables,Tables 1-5,100033,4 x 3,12,again,2026-04-10 20:37:05.992


In [17]:
# Correct and wrong answer counts
correct_wrong_ans_counts_df = fetch_df("""
SELECT last_result, COUNT(*)
FROM reviews
GROUP BY last_result
ORDER BY COUNT(*) DESC;
""")
correct_wrong_ans_counts_df

,last_result,count
0,good,964
1,again,21
